In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.svm import SVC
from sklearn.neighbors import KNeighborsClassifier
from sklearn.naive_bayes import GaussianNB
from sklearn.metrics import (accuracy_score, classification_report, confusion_matrix,
                              balanced_accuracy_score, f1_score, precision_score,
                              recall_score, ConfusionMatrixDisplay)

In [ ]:
df = pd.read_csv('Multiclass Diabetes Dataset.csv')
df.head()

In [ ]:
df.describe()

In [ ]:
print(f'Shape: {df.shape}')
print(f'Missing values: {df.isnull().sum().sum()}')
print(f'Class distribution: {df["Class"].value_counts().to_dict()}')

In [ ]:
np.random.seed(42)

x = df.drop(columns=['Class'])
y = df['Class'].values

feature_names = x.columns.tolist()
class_names = ['Non-Diabetic', 'Pre-Diabetic', 'Diabetic']

In [ ]:
# stratify ensures that the classes are split evenly in the data for both training and test data
x_train, x_test, y_train, y_test = train_test_split(x, y, test_size=0.2, random_state=42, stratify=y)

print(f'\n Training set size: {len(x_train)}')
print(f'\n Test set size: {len(x_test)}')
print(f'\n Training Class Distribution: {np.bincount(y_train)}')
print(f'\n Test Class Distribution: {np.bincount(y_test)}')

In [ ]:
scaler = StandardScaler()
x_train_scaled = scaler.fit_transform(x_train)
x_test_scaled = scaler.transform(x_test)

print(f'Original Mean = {x_train.mean(axis=0).round(2).values}')
print(f'Scaled Mean = {x_train_scaled.mean(axis=0).round(2)}')

In [ ]:
classifiers = {
    'Logistic Regression': LogisticRegression(
        solver='lbfgs',
        max_iter=1000,
        random_state=42
    ),
    'Decision Tree': DecisionTreeClassifier(
        max_depth=5,
        random_state=42
    ),
    'Random Forest': RandomForestClassifier(
        n_estimators=100,
        max_depth=5,
        random_state=42
    ),
    'Gradient Boosting': GradientBoostingClassifier(
        n_estimators=100,
        max_depth=3,
        random_state=42
    ),
    'SVC': SVC(
        kernel='rbf',
        probability=True,
        random_state=42
    ),
    'KNN': KNeighborsClassifier(
        n_neighbors=5,
        weights='distance'
    ),
    'Naive Bayes': GaussianNB()
}

In [ ]:
results = []
predictions = {}

for name, clf in classifiers.items():

    # train models
    clf.fit(x_train_scaled, y_train)

    # predict
    y_pred = clf.predict(x_test_scaled)

    # metrics
    accuracy     = accuracy_score(y_test, y_pred)
    balanced_acc = balanced_accuracy_score(y_test, y_pred)
    precision    = precision_score(y_test, y_pred, average='weighted', zero_division=0)
    recall       = recall_score(y_test, y_pred, average='weighted', zero_division=0)
    f1_macro     = f1_score(y_test, y_pred, average='macro')
    f1_weighted  = f1_score(y_test, y_pred, average='weighted')

    results.append({
        'Classifier':        name,
        'Accuracy':          accuracy,
        'Balanced Accuracy': balanced_acc,
        'Precision':         precision,
        'Recall':            recall,
        'F1 (Macro)':        f1_macro,
        'F1 (Weighted)':     f1_weighted
    })

    predictions[name] = y_pred

    print("\n")
    print(f"{name}")
    print(f"Test Accuracy: {accuracy:.4f}")

In [ ]:
print('Result Summary')
results_df = pd.DataFrame(results).sort_values('Accuracy', ascending=False)
results_df

In [ ]:
# Classification report
for name, y_pred in predictions.items():
    print(f'{name} Classification Report')
    print(classification_report(y_test, y_pred, target_names=class_names))
    print("\n")

In [ ]:
for name, y_pred in predictions.items():
    print(f'{name} Confusion Matrix')
    ConfusionMatrixDisplay.from_predictions(y_test, y_pred, display_labels=class_names)
    plt.show()
    print("\n")

## Interpretation of Results

**Q1 - Which model achieved the highest accuracy?**

Based on the results table the top performing model by accuracy is shown above. The dataset has 3 diabetes classes based on blood marker readings like HbA1c, BMI and Cholesterol levels, so models that can find non-linear patterns in those features tend to perform better.

**Q2 - Which model produced the best balance between precision and recall?**

The model with the closest precision and recall values has the best balance between the two. In diabetes classification this matters because high precision means predictions are reliable while high recall means the model is not missing real cases. A model that is too biased towards one would not be useful in practice.

**Q3 - Which model produced the highest F1-Score?**

The F1-score is the harmonic mean of precision and recall so the model ranked highest here performs most consistently across all three classes. Because Pre-Diabetic is the smallest class (40 samples) the macro F1 and weighted F1 can differ slightly — the macro F1 treats all classes equally while weighted accounts for class size.

**Q4 - What insights can be drawn from the confusion matrices?**

Looking across the confusion matrices a few things stand out. Pre-Diabetic is the hardest class to predict correctly across all models because it has the fewest samples. Most of the misclassifications happen between neighbouring classes — Non-Diabetic gets confused with Pre-Diabetic and Pre-Diabetic with Diabetic, which makes sense since these conditions exist on a spectrum rather than having a hard boundary. Models with higher accuracy show a clearer diagonal pattern in their confusion matrix with fewer off-diagonal values.

**Q5 - Which model would you recommend for this classification problem and why?**

Based on the results I would recommend Random Forest for this dataset. It builds multiple decision trees and averages their predictions which reduces overfitting compared to a single Decision Tree, which is important on a smaller dataset of 264 records. It also handles the class imbalance better than distance based models like KNN which can lean towards predicting the majority class. Features like HbA1c and BMI are well known diabetes indicators and Random Forest can rank feature importance which makes the model more interpretable. It consistently scores well across all four metrics without needing a lot of tuning.